# Q6 : combien de chambres surréserver, hôtel par hôtel

**Question de la direction** : « Si on surréserve, combien de chambres peut-on vendre en trop sans risquer de refuser un client à l'arrivée ? Un client refusé coûte bien plus cher qu'une chambre vide. »

**Auteure** : Kadidiatou Ibrahima Bagayoko

**Point de départ** : les probabilités d'annulation hors échantillon de la question 5 (`data/clean/proba_annulation.csv`).

In [2]:
import pandas as pd
import numpy as np
from scipy.stats import binom
import matplotlib.pyplot as plt

df = pd.read_csv("../data/clean/valmont_clean.csv")
df = df[df["flag_anomalie"] == 0].copy()
proba = pd.read_csv("../data/clean/proba_annulation.csv")

# On rattache la probabilité de la Q5 à chaque réservation
df = df.merge(proba[["reservation_id", "proba_annulation"]], on="reservation_id", how="left")

assert len(df) == 794
assert df["proba_annulation"].notna().all()
print("Réservations :", df.shape)
print("Hôtels :", df["hotel_id"].nunique())
print("Chambres au total :", df.drop_duplicates("hotel_id")["chambres"].sum())

Réservations : (794, 24)
Hôtels : 32
Chambres au total : 6003


## 1. Le taux d'annulation de chaque hôtel

Avec environ 25 réservations par hôtel, le taux brut est instable. On compare trois estimations :
- **taux brut** : annulations de l'hôtel divisées par ses réservations ;
- **taux lissé** : le taux brut ramené vers celui de sa catégorie, avec un poids de 20 réservations fictives au taux de la catégorie. Plus un hôtel a de réservations, plus son propre taux pèse ;
- **taux du modèle** : la moyenne des probabilités de la question 5 pour les réservations de l'hôtel.

In [3]:
# Taux d'annulation moyen de chaque catégorie, point d'appui du lissage
taux_categorie = df.groupby("categorie")["a_annule"].mean()

hotels = (df.groupby(["hotel_id", "ville", "categorie", "chambres"])
            .agg(reservations=("a_annule", "size"),
                 annulations=("a_annule", "sum"),
                 taux_modele=("proba_annulation", "mean"),
                 prix_moyen=("prix_nuit_eur", "mean"))
            .reset_index())

hotels["taux_brut"] = hotels["annulations"] / hotels["reservations"]

# Lissage : 20 réservations fictives au taux de la catégorie
m = 20
hotels["taux_lisse"] = ((hotels["annulations"] + m * hotels["categorie"].map(taux_categorie))
                        / (hotels["reservations"] + m))

print(taux_categorie.round(3))
hotels[["taux_brut", "taux_lisse", "taux_modele"]].describe().round(3)

categorie
3 étoiles    0.252
4 étoiles    0.275
5 étoiles    0.256
Name: a_annule, dtype: float64


,taux_brut,taux_lisse,taux_modele
count,32.000,32.000,32.000
mean,0.258,0.260,0.259
std,0.087,0.046,0.025
min,0.071,0.191,0.217
25%,0.198,0.225,0.236
50%,0.236,0.250,0.259
75%,0.311,0.284,0.276
max,0.500,0.376,0.309


## 2. Les écarts entre hôtels sont-ils réels ou dus au hasard ?

Si tous les hôtels avaient le même risque d'annulation (26 %), le hasard seul ferait varier leurs taux observés. Avec environ 25 réservations par hôtel, l'écart-type attendu sous ce seul effet du hasard vaut racine de p(1 - p) / n. On le compare à l'écart-type réellement observé entre les 32 hôtels.

In [4]:
p_global = df["a_annule"].mean()
n_moyen = hotels["reservations"].mean()

ecart_hasard = np.sqrt(p_global * (1 - p_global) / n_moyen)
ecart_observe = hotels["taux_brut"].std()

print("Réservations moyennes par hôtel :", round(n_moyen, 1))
print("Écart-type attendu par le seul hasard :", round(ecart_hasard, 3))
print("Écart-type observé entre les hôtels   :", round(ecart_observe, 3))

Réservations moyennes par hôtel : 24.8
Écart-type attendu par le seul hasard : 0.088
Écart-type observé entre les hôtels   : 0.087


**Lecture des résultats et choix du taux**

- L'écart-type observé entre les hôtels (0,087) est égal à celui que produirait le seul hasard sur 25 réservations (0,088). **Les écarts de taux d'annulation entre hôtels ne traduisent aucune différence réelle** : un hôtel à 7 % et un hôtel à 50 % ont, selon toute vraisemblance, le même risque de fond.
- Ce résultat rejoint la question 5 : la catégorie, la taille ou la ville de l'hôtel n'amélioraient pas la prédiction de l'annulation.
- Appliquer une règle de surréservation sur le taux brut serait donc dangereux : on surréserverait fortement l'hôtel à 50 %, sur la foi d'un simple hasard, au risque de refuser des clients.

**Taux retenu : le taux du modèle de la question 5** (de 21,7 % à 30,9 % selon l'hôtel). Il ne varie qu'en fonction de la seule différence démontrée entre hôtels : leur proportion de réservations lointaines. Le taux lissé, compris entre les deux, sert de contrôle.